# Ropedia Academy — C3 · Video understanding backbones: SlowFast & VideoMAE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/C3.ipynb)

> **Contrasts both video paradigms in one run: SlowFast's two-rate pathways with lateral fusion, and VideoMAE's high-ratio tube masking that forces real spatio-temporal learning.**
>
> 一次运行对比两种视频范式：SlowFast 的双速率通路加横向融合，与 VideoMAE 高比例管状掩码逼出真正的时空学习。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/C3

In [ ]:
import torch, torch.nn as nn

clip = torch.randn(1, 3, 32, 56, 56)                  # (B, C, T, H, W): a 32-frame clip

# SlowFast: TWO pathways at different frame rates, fused by a lateral connection.
slow = clip[:, :, ::8]                                 # few frames, full detail (T=4)
fast = clip[:, :, ::2]                                 # many frames, low capacity (T=16)
slow_net, fast_net = nn.Conv3d(3, 64, 1), nn.Conv3d(3, 8, 1)   # fast = fewer channels
lateral = nn.Conv3d(8, 64, 1)                          # fuse fast -> slow
fused = slow_net(slow) + lateral(fast_net(fast)[:, :, ::4])     # align time, add
print("SlowFast fused feature:", tuple(fused.shape))

# VideoMAE: TUBE masking - hide the SAME spatial patches across all frames, at a
# HIGH ratio (~90%) because video is temporally redundant.
T, N = 16, 14*14
keep = int(0.10 * N)
vis = torch.randperm(N)[:keep]
mask = torch.ones(T, N, dtype=torch.bool); mask[:, vis] = False    # one tube, all frames
print(f"masked {mask.float().mean()*100:.0f}% of patches; visible: {keep}/{N}")
print("high ratio kills the 'copy a neighbouring frame' shortcut -> real understanding")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/C3
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks